In [1]:
import os
import sys
import json

sys.path.append(
    os.path.abspath("..")
)

from pyspark.sql import SparkSession

from heal import CodeMaster
from dataio import DataWriter

In [2]:
spark = (

    SparkSession.builder

    .appName(
        "SelfHealingPipeline"
    )

    .master("local[*]")

    .getOrCreate()

)

print("Spark Started")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/23 12:39:54 WARN Utils: Your hostname, Vedants-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.72.231.251 instead (on interface en0)
26/05/23 12:39:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/23 12:39:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Started


In [3]:
with open(
        "../config/test_rules.json"
) as f:

    rules = json.load(f)


with open(
        "../config/test_source.json"
) as f:

    source = json.load(f)


print("Configs loaded")

Configs loaded


In [4]:
csv_path = source["source"]["path"]

df = (
    spark.read
    .option(
        "header",
        True
    )
    .option(
        "inferSchema",
        True
    )
    .csv(csv_path)

)

print(
    "\nInput Data"
)

df.show(
    truncate=False
)


Input Data
+--------+-----------+------------------+------+---------+-------------------+
|order_id|customer_id|customer_email    |amount|status   |created_at         |
+--------+-----------+------------------+------+---------+-------------------+
|1001    |C001       |john@example.com  |250.5 |completed|2026-05-20 10:30:00|
|1002    |C002       |sarah@gmail.com   |1500.0|pending  |2026-05-21 09:00:00|
|1003    |C003       |mike@yahoo.com    |99.99 |completed|2026-05-18 14:20:00|
|1004    |C004       |invalid_email     |450.25|completed|2026-05-22 11:00:00|
|1005    |C005       |anna@gmail.com    |-500.0|completed|2026-05-19 08:00:00|
|1006    |C006       |bob@gmail         |100.0 |completed|2026-05-23 12:00:00|
|1007    |NULL       |jenny@gmail.com   |700.0 |pending  |2026-05-20 15:00:00|
|NULL    | C008      |mark@gmail.com    |800.0 |completed|2026-05-21 16:00:00|
|1008    |C009       |alex@gmail.com    |1.0E7 |completed|2026-05-20 09:00:00|
|1009    |C010       |lisa@gmail.com    

In [5]:
pipeline = CodeMaster(

        rules_dict=rules,

        input_data=df,

        spark=spark

)

results = pipeline.run()

2026-05-23 12:40:04,376 - INFO - HealData initialized with status columns
2026-05-23 12:40:04,377 - INFO - Pipeline started
2026-05-23 12:40:04,377 - INFO - Applying rule rule_001 - order_id_not_null
2026-05-23 12:40:04,378 - INFO - [rule_001] Applying not_null check on field: 'order_id'
2026-05-23 12:40:04,399 - INFO - [rule_001] Null rows on 'order_id' marked as dead
2026-05-23 12:40:04,399 - INFO - Applying rule rule_002 - customer_id_not_null
2026-05-23 12:40:04,400 - INFO - [rule_002] Applying not_null check on field: 'customer_id'
2026-05-23 12:40:04,416 - INFO - [rule_002] Null rows on 'customer_id' marked as dead
2026-05-23 12:40:04,416 - INFO - Applying rule rule_003 - amount_range
2026-05-23 12:40:04,416 - INFO - [rule_003] Applying range validation on field 'amount' between 0.01 and 999999.99
2026-05-23 12:40:04,449 - INFO - [rule_003] Range failures on 'amount' healed via clamp
2026-05-23 12:40:04,449 - INFO - Applying rule rule_004 - email_regex
2026-05-23 12:40:04,449 - I

In [6]:
clean_df = results[0]

healed_df = results[1]

rejected_df = results[2]

In [7]:
print(
"\n========== CLEAN =========="
)

clean_df.show(
truncate=False
)


print(
"\n========== HEALED =========="
)

healed_df.show(
truncate=False
)


print(
"\n========== REJECTED =========="
)

rejected_df.show(
truncate=False
)



========== CLEAN ==========
+--------+-----------+----------------+---------+---------+-------------------+-------+---------+
|order_id|customer_id|customer_email  |amount   |status   |created_at         |_status|_heal_log|
+--------+-----------+----------------+---------+---------+-------------------+-------+---------+
|1001    |C001       |john@example.com|250.5    |completed|2026-05-20 10:30:00|correct|NULL     |
|1002    |C002       |sarah@gmail.com |1500.0   |pending  |2026-05-21 09:00:00|correct|NULL     |
|1003    |C003       |mike@yahoo.com  |99.99    |completed|2026-05-18 14:20:00|correct|NULL     |
|1004    |C004       |NULL            |450.25   |completed|2026-05-22 11:00:00|correct|NULL     |
|1005    |C005       |anna@gmail.com  |0.01     |completed|2026-05-19 08:00:00|correct|NULL     |
|1008    |C009       |alex@gmail.com  |999999.99|completed|2026-05-20 09:00:00|correct|NULL     |
|1009    |C010       |lisa@gmail.com  |0.01     |completed|2026-05-18 12:00:00|correct|NU

In [8]:
writer = DataWriter(

        config=source,

        spark=spark

)


writer.data_write(
    clean_df,
    "clean"
)

writer.data_write(
    healed_df,
    "healed"
)

writer.data_write(
    rejected_df,
    "rejected"
)


print(
"\nFiles written successfully"
)


###########################################
# Verify Saved Outputs
###########################################

saved = spark.read.csv(

        "../output/healed",

        header=True

)

print(
"\nSaved healed data:"
)

saved.show(
truncate=False
)


###########################################
# Stop Spark
###########################################

spark.stop()

print(
"\nPipeline Completed"
)

2026-05-23 12:40:25,793 - INFO - Writing clean dataframe
2026-05-23 12:40:25,794 - INFO - Writing to: ../output/clean
2026-05-23 12:40:26,011 - INFO - clean written successfully
2026-05-23 12:40:26,012 - INFO - Writing to: ../output/clean
2026-05-23 12:40:26,137 - INFO - clean written successfully
2026-05-23 12:40:26,138 - INFO - Writing to: ../output/clean
2026-05-23 12:40:26,256 - INFO - clean written successfully
2026-05-23 12:40:26,256 - INFO - Writing healed dataframe
2026-05-23 12:40:26,256 - INFO - Writing to: ../output/healed
2026-05-23 12:40:26,378 - INFO - healed written successfully
2026-05-23 12:40:26,378 - INFO - Writing to: ../output/healed
2026-05-23 12:40:26,486 - INFO - healed written successfully
2026-05-23 12:40:26,486 - INFO - Writing to: ../output/healed
2026-05-23 12:40:26,594 - INFO - healed written successfully
2026-05-23 12:40:26,594 - INFO - Writing rejected dataframe
2026-05-23 12:40:26,594 - INFO - Writing to: ../output/rejected
2026-05-23 12:40:26,719 - INF


Files written successfully

Saved healed data:
+----+----+------------------+-----+---------+-----------------------------+------+---------------------------------------------------------------------------------------------------+
|1006|C006|_c2               |100.0|completed|2026-05-23T12:40:26.494+05:30|healed|rule_005|future_timestamp: field 'created_at' was future timestamp, replaced with current timestamp|
+----+----+------------------+-----+---------+-----------------------------+------+---------------------------------------------------------------------------------------------------+
|1010|C011|tom@gmail.com     |450.0|completed|2026-05-23T12:40:26.494+05:30|healed|rule_005|future_timestamp: field 'created_at' was future timestamp, replaced with current timestamp|
|1011|C012|emma_new@gmail.com|350.0|completed|2026-05-22T14:00:00.000+05:30|healed|NULL                                                                                               |
+----+----+------------------+--